# Notebook 8 — Bayesian Hierarchical Safety Model

**Problem:** Some segments have few observations — their sample mean TTC
is unreliable.  Pooling all segments ignores real variation.  Hierarchical
Bayesian modelling finds the optimal balance (partial pooling).

**Model:**
```
μ_pop   ~ Normal(5, 5²)        # population mean TTC
σ_pop   ~ HalfNormal(2)        # between-segment SD
μⱼ      ~ Normal(μ_pop, σ_pop) # per-segment mean (partial pooling)
y_ij    ~ Normal(μⱼ, σ_obs)    # frame-level TTC observations
```

Fitted via Metropolis-Hastings MCMC (pure numpy — no external PPL).

Outputs:
- Posterior distributions for population and segment-level TTC
- Per-segment risk probability: P(μⱼ < 3 s)
- Shrinkage visualisation
- Posterior predictive checks

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.waymo_loader import load_dataset, extract_trajectories
from src.metrics.safety_metrics import batch_ttc
from src.bayesian.hierarchical_model import (
    fit_hierarchical_model, posterior_predictive_check,
    plot_hierarchical_posteriors
)

sns.set_theme(style='whitegrid')

In [2]:
dataset = load_dataset('../data', max_segments=8)
trajectories = extract_trajectories(dataset['lidar_box'])
ttc_df = batch_ttc(trajectories)

# Replace inf with large value; we model finite-TTC frames
ttc_df['min_ttc_obs'] = ttc_df['min_ttc'].replace(np.inf, np.nan)
ttc_finite = ttc_df.dropna(subset=['min_ttc_obs'])

print(f'Segments: {ttc_finite["segment_id"].nunique()}')
print(f'Observations: {len(ttc_finite):,}')
ttc_finite.groupby('segment_id')['min_ttc_obs'].describe().round(2)

Segments: 8
Observations: 1,563


/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: invalid value encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: divide by zero encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)


,count,mean,std,min,25%,50%,75%,max
segment_id,,,,,,,,
10203656353524179475_7625_000_7645_000,197.0,0.25,0.45,0.0,0.00,0.00,0.38,2.80
1024360143612057520_3580_000_3600_000,198.0,0.02,0.09,0.0,0.00,0.00,0.00,0.77
10247954040621004675_2180_000_2200_000,197.0,84.45,56.73,0.0,47.09,78.98,114.33,249.28
10289507859301986274_4200_000_4220_000,197.0,0.29,0.33,0.0,0.00,0.14,0.54,1.24
10335539493577748957_1372_870_1392_870,197.0,39.15,58.58,0.1,0.39,4.86,63.75,235.43
10359308928573410754_720_000_740_000,198.0,3.78,8.52,0.3,0.48,0.68,1.22,45.49
10448102132863604198_472_000_492_000,182.0,0.50,0.32,0.0,0.28,0.42,0.72,1.29
10689101165701914459_2072_300_2092_300,197.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00


## Fit Hierarchical Model via MCMC

In [3]:
print('Running Metropolis-Hastings MCMC (this may take ~30s)...')
posterior = fit_hierarchical_model(
    ttc_finite,
    metric_col='min_ttc_obs',
    segment_col='segment_id',
    n_samples=2000,
    warmup=1000,
    step_size=0.12,
)
print(f'Acceptance rate: {posterior.acceptance_rate:.2%}  (target: 20–50%)')
print(f'\nPopulation-level summary:')
for k, v in posterior.population_summary().items():
    print(f'  {k}: {v}')

Running Metropolis-Hastings MCMC (this may take ~30s)...


Acceptance rate: 4.05%  (target: 20–50%)

Population-level summary:
  mu_pop_mean: 16.281397969691785
  mu_pop_std: 0.3253689155152774
  mu_pop_95ci: (15.431699246954985, 16.805174594983793)
  sigma_pop_mean: 12.511976468970106
  shrinkage_factor: 0.9972887539917343


In [4]:
fig = plot_hierarchical_posteriors(posterior)
plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_28610/3654294237.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Per-Segment Risk: P(μⱼ < 3 s)

In [5]:
risk = posterior.risk_probability(threshold=3.0)
risk_df = risk.reset_index()
risk_df.columns = ['segment_id', 'risk_prob']
risk_df = risk_df.sort_values('risk_prob', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tomato' if r > 0.3 else 'steelblue' for r in risk_df['risk_prob']]
ax.barh(range(len(risk_df)), risk_df['risk_prob'], color=colors, edgecolor='white')
ax.set_yticks(range(len(risk_df)))
ax.set_yticklabels([s[:25] for s in risk_df['segment_id']], fontsize=8)
ax.axvline(0.3, color='red', linestyle='--', label='Risk threshold (30%)')
ax.set_xlabel('P(μⱼ < 3 s)')
ax.set_title('Posterior Risk Probability per Segment')
ax.legend()
plt.tight_layout()
plt.show()

print('High-risk segments (P > 0.3):')
print(risk_df[risk_df['risk_prob'] > 0.3].to_string(index=False))

High-risk segments (P > 0.3):
                            segment_id  risk_prob
10203656353524179475_7625_000_7645_000        1.0
 1024360143612057520_3580_000_3600_000        1.0
10289507859301986274_4200_000_4220_000        1.0
  10448102132863604198_472_000_492_000        1.0
10689101165701914459_2072_300_2092_300        1.0


/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_28610/1313097832.py:15: UserWarning: Glyph 11388 (\N{LATIN SUBSCRIPT SMALL LETTER J}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_28610/1313097832.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Posterior Predictive Check

In [6]:
observations = [
    ttc_finite[ttc_finite['segment_id'] == seg]['min_ttc_obs'].values
    for seg in posterior.segment_ids
]
ppc = posterior_predictive_check(posterior, observations)

print(f'80% posterior predictive coverage: {ppc["coverage_80"]:.2%}')
print('(should be near 80% for a well-calibrated model)')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(ppc['observed_means'], ppc['ppc_means'], s=60, color='steelblue', zorder=3)
lim = [min(ppc['observed_means'].min(), ppc['ppc_means'].min()),
       max(ppc['observed_means'].max(), ppc['ppc_means'].max())]
ax.plot(lim, lim, 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Observed Segment Mean TTC (s)')
ax.set_ylabel('Posterior Predictive Mean TTC (s)')
ax.set_title('Posterior Predictive Check')
ax.legend()
plt.tight_layout()
plt.show()

80% posterior predictive coverage: 100.00%
(should be near 80% for a well-calibrated model)


/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_28610/3148176242.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
